# 03 - Sistema especialista com regras SE–ENTÃO

A abordagem baseada em regras realiza previsões de Employability_Score e Placement_Status por meio de uma base de conhecimento externa ao motor de inferência. As regras heurísticas são armazenadas na pasta rules; o motor de inferência, implementado em src/expert_system.py, aplica suas condições em sequência. O conjunto de teste é utilizado apenas na avaliação.

In [1]:
from pathlib import Path
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
ROOT = Path.cwd().resolve()
if not (ROOT/'data').exists(): ROOT = ROOT.parent
assert (ROOT/'src').exists(), 'Abra o notebook a partir da pasta do projeto.'
sys.path.insert(0,str(ROOT))
TAB=ROOT/'reports'/'tables'; FIG=ROOT/'reports'/'figures'
META=json.loads((ROOT/'data/processed/metadata.json').read_text(encoding='utf-8'))
print('Projeto:', ROOT)

Projeto: C:\Users\Felipe Erik\Desktop\Trabalho2IA\final\student_career_ia


In [2]:
from sklearn.model_selection import train_test_split

def load_split(task):
    df=pd.read_csv(ROOT/'data/processed'/f'dataset_{task}.csv')
    target=META[task]['target']; X=df.drop(columns=target); y=df[target]
    if task=='classification': y=(y=='Placed').astype(int)
    idx_train,idx_test=train_test_split(np.arange(len(df)), test_size=.2,random_state=42, stratify=y if task=='classification' else None)
    return X.iloc[idx_train],X.iloc[idx_test],y.iloc[idx_train],y.iloc[idx_test]

## Base de conhecimento e motor de inferência
O motor avalia as condições em sequência, com política primeira regra correspondente e valor padrão quando nenhuma regra é satisfeita. A base de conhecimento está armazenada em rules/, enquanto src/expert_system.py implementa o motor de inferência e o carregamento. As regras são avaliadas na ordem do arquivo. Cada regra é classificada como heurística; isso não substitui a justificativa do conhecimento de domínio.

## Verificação da cobertura dos intervalos de entrevista

Na regressão, a base fixa utiliza cinco intervalos: nota inferior a 67; de 67 a menos de 75; de 75 a menos de 82; de 82 a menos de 88; e nota a partir de 88. Os intervalos são mutuamente exclusivos e cobrem todos os valores numéricos não ausentes. A tabela seguinte compara as saídas das regras com a média observada de empregabilidade somente no treino; ela descreve uma linha de base, sem empregar o teste para escolher limiares.

In [3]:
X_tr, _, y_tr, _ = load_split('regression')
faixas = [-np.inf, 67, 75, 82, 88, np.inf]
rotulos = ['R1: <67', 'R2: [67,75)', 'R3: [75,82)', 'R4: [82,88)', 'R5: >=88']
valores_regras = [177.5579871348164, 203.02669966137015, 216.83369479272565,
                 227.4591194519766, 240.40362777777776]
tabela_treino = X_tr[['Interview_Score']].copy()
tabela_treino['Employability_Score'] = y_tr.to_numpy()
tabela_treino['intervalo'] = pd.cut(tabela_treino['Interview_Score'],
                                   bins=faixas, labels=rotulos, right=False)
resumo = (tabela_treino.groupby('intervalo', observed=False)['Employability_Score']
          .agg(n='count', media='mean', mediana='median', desvio='std', minimo='min', maximo='max'))
resumo['previsao_regra'] = valores_regras
resumo['diferenca_media'] = resumo['previsao_regra'] - resumo['media']
display(resumo.round(3))
print('Valores de entrevista ausentes no treino:', tabela_treino['Interview_Score'].isna().sum())
print('Amostras não atribuídas às faixas:', tabela_treino['intervalo'].isna().sum())

,n,media,mediana,desvio,minimo,maximo,previsao_regra,diferenca_media
intervalo,,,,,,,,
R1: <67,7462,177.558,178.25,21.183,82.30,249.70,177.558,0.0
"R2: [67,75)",7678,203.027,203.10,16.134,145.20,258.80,203.027,0.0
"R3: [75,82)",8853,216.834,217.25,14.830,156.65,268.50,216.834,0.0
"R4: [82,88)",7007,227.459,227.85,13.914,175.55,272.55,227.459,0.0
R5: >=88,9000,240.404,241.10,13.579,188.60,279.05,240.404,0.0


Valores de entrevista ausentes no treino: 0
Amostras não atribuídas às faixas: 0


### Análise dos limites

- Os operadores < e >= determinam a qual regra pertencem os valores exatamente iguais a 67, 75, 82 e 88; portanto, não há interseção entre intervalos adjacentes.
- A proximidade entre a média observada e a previsão da regra pode ser examinada na coluna diferenca_media. O desvio padrão mostra a dispersão que uma previsão fixa por faixa não captura.
- A avaliação de generalização permanece separada: mudanças nas regras só podem se basear em informações do treino ou conhecimento justificado do domínio.

In [4]:
from src.expert_system import load_rules
for task in ['regression','classification']:
    Xtr,Xte,ytr,yte=load_split(task)
    expert=load_rules(task)
    print(f'\n{task}: {len(expert.rules)} regras, padrão={expert.defaults}')
    for i,r in enumerate(expert.rules[:6],1):
        print(i, 'TIPO:',r['type'],'SE',r['conditions'],'ENTÃO',r['value'])
    print('Primeiras previsões:', expert.predict(Xte.head(5)))


regression: 5 regras, padrão=177.5579871348164
1 TIPO: heuristica SE [('Interview_Score', '<', 67.0)] ENTÃO 177.5579871348164
2 TIPO: heuristica SE [('Interview_Score', '>=', 67.0), ('Interview_Score', '<', 75.0)] ENTÃO 203.02669966137015
3 TIPO: heuristica SE [('Interview_Score', '>=', 75.0), ('Interview_Score', '<', 82.0)] ENTÃO 216.83369479272565
4 TIPO: heuristica SE [('Interview_Score', '>=', 82.0), ('Interview_Score', '<', 88.0)] ENTÃO 227.4591194519766
5 TIPO: heuristica SE [('Interview_Score', '>=', 88.0)] ENTÃO 240.40362777777776
Primeiras previsões: [216.83369479 240.40362778 216.83369479 216.83369479 240.40362778]

classification: 6 regras, padrão=0
1 TIPO: heuristica SE [('Interview_Score', '>=', 80), ('Programming_Skill', '>=', 7)] ENTÃO 1
2 TIPO: heuristica SE [('Employability_Score', '>=', 238.8)] ENTÃO 1
3 TIPO: heuristica SE [('Employability_Score', '>=', 227.0), ('Employability_Score', '<', 238.8)] ENTÃO 1
4 TIPO: heuristica SE [('Employability_Score', '>=', 213.8), 

## Desempenho de treino versus teste
Os ajustes da base de regras são restritos ao conjunto de treinamento. As métricas apresentadas abaixo foram calculadas na execução com 50.000 linhas e divisão 80/20.

In [5]:
m=pd.read_csv(TAB/'model_metrics.csv')
display(m.loc[m.model=='Sistema_Especialista'].round(4))
g=pd.read_csv(TAB/'generalization.csv');display(g.loc[g.model=='Sistema_Especialista'].round(4))

,task,model,split,MAE,MSE,RMSE,MAPE,R2,MedAE,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,regression,Sistema_Especialista,train,12.6619,257.7171,16.0536,6.2169,0.6415,10.6036,NaN,NaN,NaN,NaN,NaN,NaN
1,regression,Sistema_Especialista,test,12.5218,253.1047,15.9093,6.1618,0.6494,10.4442,NaN,NaN,NaN,NaN,NaN,NaN
10,classification,Sistema_Especialista,train,NaN,NaN,NaN,NaN,NaN,NaN,0.8051,0.8658,0.888,0.8768,NaN,NaN
11,classification,Sistema_Especialista,test,NaN,NaN,NaN,NaN,NaN,NaN,0.8000,0.8625,0.885,0.8736,NaN,NaN


,task,model,metric,train,test,gap_train_minus_test
4,classification,Sistema_Especialista,F1,0.8768,0.8736,0.0032
9,regression,Sistema_Especialista,RMSE,16.0536,15.9093,0.1443


## Registro do esforço de elaboração

A elaboração da base de regras é documentada em três sessões para cada tarefa. Registram-se datas, horários, duração e quantidade de regras por sessão, independentemente do custo de inferência computacional.

In [6]:
log=pd.read_csv(ROOT/'logs/cronometro_regras.csv')
display(log)
if len(log):
    for task,part in log.groupby('tarefa'):
        t=pd.to_numeric(part['minutos'],errors='coerce').dropna();r=pd.to_numeric(part['regras_criadas'],errors='coerce').dropna()
        print(task, {'sessoes':len(part),'tempo_total_min':round(t.sum(),2),'media_sessao_min':round(t.mean(),2),'desvio_padrao_min':round(t.std(ddof=1),2) if len(t)>1 else None, 'min_por_regra':round(t.sum()/r.sum(),2) if r.sum()>0 else None})
else: print('Não há sessões de elaboração humana registradas.')

,tarefa,sessao,data,hora_inicio,hora_fim,minutos,regras_criadas,interrupcoes,observacoes
0,regression,1,2026-10-26,07:15:00,07:47:34,32.566667,2,Nao registrado,NaN
1,regression,2,2026-10-26,07:55:00,08:12:48,17.800000,1,Nao registrado,NaN
2,regression,3,2026-10-26,08:17:00,08:38:12,21.200000,2,Nao registrado,NaN
3,classification,1,2026-10-26,08:45:00,09:17:26,32.433333,3,Nao registrado,NaN
4,classification,2,2026-10-26,09:21:00,09:42:57,21.950000,2,Nao registrado,NaN
5,classification,3,2026-10-26,09:49:00,10:03:18,14.300000,1,Nao registrado,NaN


classification {'sessoes': 3, 'tempo_total_min': np.float64(68.68), 'media_sessao_min': np.float64(22.89), 'desvio_padrao_min': np.float64(9.1), 'min_por_regra': np.float64(11.45)}
regression {'sessoes': 3, 'tempo_total_min': np.float64(71.57), 'media_sessao_min': np.float64(23.86), 'desvio_padrao_min': np.float64(7.73), 'min_por_regra': np.float64(14.31)}


## Discussão
As regras utilizam um pequeno número de condições e não aprendem continuamente com novos exemplos. Sua interpretação costuma ser simples, mas manutenção e cobertura podem ser difíceis. Na regressão, previsões por faixas criam degraus; em classificação, regras por limiar perdem nuances. Cuidado: Employability_Score aparece como feature da classificação; é necessário confirmar sua disponibilidade no momento de previsão e que não incorpora Placement_Status.

## Cronometragem da base de conhecimento

Os registros de elaboração estão disponíveis em logs/cronometro_regras.csv. A duração de cada sessão e as estatísticas de esforço são apresentadas a seguir. Não foram registrados detalhes de interrupções.

In [7]:
from pathlib import Path
import pandas as pd
raiz = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
display(pd.read_csv(raiz / "logs/cronometro_regras.csv"))
display(pd.read_csv(raiz / "reports/tables/estatisticas_cronometragem.csv"))

,tarefa,sessao,data,hora_inicio,hora_fim,minutos,regras_criadas,interrupcoes,observacoes
0,regression,1,2026-10-26,07:15:00,07:47:34,32.566667,2,Nao registrado,NaN
1,regression,2,2026-10-26,07:55:00,08:12:48,17.800000,1,Nao registrado,NaN
2,regression,3,2026-10-26,08:17:00,08:38:12,21.200000,2,Nao registrado,NaN
3,classification,1,2026-10-26,08:45:00,09:17:26,32.433333,3,Nao registrado,NaN
4,classification,2,2026-10-26,09:21:00,09:42:57,21.950000,2,Nao registrado,NaN
5,classification,3,2026-10-26,09:49:00,10:03:18,14.300000,1,Nao registrado,NaN


,tarefa,sessoes,regras_reportadas,tempo_total_min,media_sessao_min,desvio_padrao_sessao_min,tempo_medio_por_regra_min,origem
0,regression,3,5,52.566667,17.522222,0.302459,10.513333,informacao declarada pelo estudante; nao auditada
1,classification,3,6,52.683333,17.561111,1.396855,8.780556,informacao declarada pelo estudante; nao auditada


## Sessões de elaboração

A tabela apresenta os horários, durações e quantidades de regras registrados para cada tarefa.

In [8]:
from pathlib import Path
import pandas as pd
BASE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
display(pd.read_csv(BASE/'logs/cronometro_regras.csv'))
display(pd.read_csv(BASE/'reports/tables/cronometro_resumo.csv'))

,tarefa,sessao,data,hora_inicio,hora_fim,minutos,regras_criadas,interrupcoes,observacoes
0,regression,1,2026-10-26,07:15:00,07:47:34,32.566667,2,Nao registrado,NaN
1,regression,2,2026-10-26,07:55:00,08:12:48,17.800000,1,Nao registrado,NaN
2,regression,3,2026-10-26,08:17:00,08:38:12,21.200000,2,Nao registrado,NaN
3,classification,1,2026-10-26,08:45:00,09:17:26,32.433333,3,Nao registrado,NaN
4,classification,2,2026-10-26,09:21:00,09:42:57,21.950000,2,Nao registrado,NaN
5,classification,3,2026-10-26,09:49:00,10:03:18,14.300000,1,Nao registrado,NaN


,tarefa,numero_sessoes,regras_criadas,tempo_total_minutos,tempo_medio_sessao_minutos,desvio_padrao_amostral_minutos,tempo_medio_por_regra_minutos
0,regression,3,5,71.566667,23.855556,7.733214,14.313333
1,classification,3,6,68.683333,22.894444,9.103484,11.447222
